# 01 · Pipeline check with pretrained SmolVLA models (Step A3)

| model | expected | what it proves |
|---|---|---|
| `smolvla_base` | ~0% success, arm moves | plumbing works end to end |
| `smolvla_libero` (`HuggingFaceVLA/smolvla_libero`) | high success | our image flip, state, keys, normalization and gripper sign are right |

Run `00_colab_setup.ipynb` once first (it downloads the demos). A100 recommended; a T4 works but is slower.
Weights are downloaded to `MyDrive/groundedvla/models/` the first time, then copied to local disk each session.

In [1]:
# ---- edit once ----
REPO_URL = "https://github.com/kaustubhadhe1206/GroundedVLA.git"   # public repo, no token needed
BRANCH = "main"

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os, subprocess
REPO = "/content/GroundedVLA"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
!git log -1 --format="commit %h  %s"

commit d286574  colab_install: don't import libero before its config exists (blocks on a prompt)


In [5]:
!apt-get -qq install -y libegl1 libgl1 > /dev/null
%cd /content/GroundedVLA
!git pull --ff-only && git log -1 --format="commit %h  %s"
!bash scripts/colab_install.sh requirements.txt

/content
Already up to date.
commit d286574  colab_install: don't import libero before its config exists (blocks on a prompt)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for groundedvla (pyproject.toml) ... done
[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.13/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
LIBERO stack OK, mujoco 3.3.2


In [6]:
from pathlib import Path
from groundedvla.utils.colab import setup_session, stage_to_local
paths = setup_session()          # env vars (EGL, caches on local disk), Drive folders, LIBERO config

profile=colab root=/content/drive/MyDrive/groundedvla commit=d286574


In [7]:
# Fast local copies of weights already on Drive (first run: run_sim downloads them to Drive)
for folder in ["smolvla_base", "smolvla_libero"]:
    if (Path(paths["models"]) / folder / "config.json").exists():
        stage_to_local(f"models/{folder}")

## Rollouts (10 per task, videos saved to Drive)

`smolvla_base` only on T1 (plumbing, ~0%). `smolvla_libero` on all 7 tasks: this is the reference for our V1 baseline. Tasks that already have 10 logged episodes on Drive are skipped.

In [9]:
!pip install -q num2words==0.5.14

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import json
EPISODES = 10
RUNS = [("smolvla_base", ["T1"]),                                      # plumbing only (~0%)
        ("smolvla_libero", ["T1", "T2", "T3", "T4", "T5", "T6", "T7"])]
SKIP_DONE = True     # skip model/task pairs that already have EPISODES results on Drive

def n_done(model, task):
    log = Path(paths["outputs"], "run_sim", model, f"{task}.jsonl")
    if not log.exists():
        return 0
    return len({json.loads(l)["episode_idx"] for l in log.read_text().splitlines() if l.strip()})

for model, tasks in RUNS:
    for task in tasks:
        if SKIP_DONE and n_done(model, task) >= EPISODES:
            print(f"skip {model} {task}: {n_done(model, task)} episodes already logged")
            continue
        !python scripts/run_sim.py {model} --task {task} --episodes {EPISODES} --viewer none --save-video

Optional: camera robustness preview (yaw ±20°, as in the light eval of T5–T7), and execution modes.

In [ ]:
# for task in ["T5", "T6", "T7"]:
#     for yaw in [20, -20]:
#         !python scripts/run_sim.py smolvla_libero --task {task} --episodes 10 --viewer none --camera-yaw {yaw} --save-video
# !python scripts/run_sim.py smolvla_libero --task T1 --episodes 10 --viewer none --horizon 1
# !python scripts/run_sim.py smolvla_libero --task T1 --episodes 10 --viewer none --mode temporal_ensemble

In [14]:
import json
from pathlib import Path
from IPython.display import Video, display, Markdown

MODEL = "smolvla_libero"
TASKS = ["T1", "T2", "T3", "T4", "T5", "T6", "T7"]   # or e.g. ["T5_yaw20"]
MAX_PER_TASK = 10

root = Path(paths["outputs"], "run_sim", MODEL)
for task in TASKS:
    # result of each episode from the log (a later run of the same episode wins)
    result = {}
    log = root / f"{task}.jsonl"
    if log.exists():
        for line in log.read_text().splitlines():
            if line.strip():
                r = json.loads(line)
                result[r["episode_idx"]] = r
    videos = sorted(root.glob(f"videos/{task}_ep*.mp4"))[:MAX_PER_TASK]
    display(Markdown(f"### {task}: {len(videos)} videos"))
    for v in videos:
        ep = int(v.stem.split("_ep")[1])
        r = result.get(ep, {})
        label = "SUCCESS" if r.get("success") else f"fail ({r.get('failure')})" if r else "?"
        print(f"{v.name}  {label}  steps={r.get('steps')}")
        display(Video(str(v), embed=True, width=512))


### T1: 10 videos

T1_ep000.mp4  SUCCESS  steps=83


T1_ep001.mp4  fail (failed_grasp)  steps=220


T1_ep002.mp4  fail (failed_grasp)  steps=220


T1_ep003.mp4  SUCCESS  steps=103


T1_ep004.mp4  fail (failed_grasp)  steps=220


... (23 outputs left out)

T3_ep006.mp4  SUCCESS  steps=95


T3_ep007.mp4  SUCCESS  steps=87


T3_ep008.mp4  SUCCESS  steps=89


T3_ep009.mp4  SUCCESS  steps=94


In [ ]:
import json
import pandas as pd
rows = []
for f in Path(paths["outputs"], "run_sim").glob("*/*.jsonl"):
    for l in f.read_text().splitlines():
        if l.strip():
            rows.append({**json.loads(l), "model": f.parent.name, "run": f.stem})   # run = T3, T3_yaw20, ...
df = pd.DataFrame(rows).drop_duplicates(["model", "run", "episode_idx"], keep="last")   # a rerun replaces
df.groupby(["model", "run"]).agg(episodes=("success", "size"), success=("success", "mean"),
                                 steps=("steps", "mean"), policy_call_ms=("latency_ms_median", "median"))